### Import Dependencies

In [2]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, FusionQuery

import psycopg2
from psycopg2.extras import RealDictCursor
import numpy as np
import os

In [3]:
from dotenv import load_dotenv
load_dotenv("../../.env")

True

### Add to Shopping Cart Tool

In [10]:
items = [
    {
        "product_id": "B01DJI7TP8",
        "quantity": 2
    },
    {
        "product_id": "B07M6SWKLT",
        "quantity": 4
    }
]

In [13]:
def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:

    """Add a list of provided items to the shopping cart.
    
    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.
        
    Returns:
        A list of the items added to the shopping cart.
    """

    conn = psycopg2.connect(
        host="aws-0-ap-southeast-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.vtluslyavpfwprmqmzft",
        password="ewTFDFew"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
        for item in items:
            product_id = item['product_id']
            quantity = item['quantity']

            qdrant_client = QdrantClient(
                url=os.getenv("QDRANT_CLOUD_URL"),
                api_key=os.getenv("QDRANT_API_KEY"),
                timeout=60
            )

            dummy_vector = np.zeros(1536).tolist()
            payload = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                        must=[
                            FieldCondition(
                                key="parent_asin",
                                match=MatchValue(value=product_id)
                            )
                        ]
                    ),
                        using="text-embedding-3-small",
                        limit=20
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            ).points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = 'USD'
        
            # Check if item already exists
            check_query = """
                SELECT id, quantity, price 
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()
            
            if existing_item:
                # Update existing item
                new_quantity = existing_item['quantity'] + quantity
                
                update_query = """
                    UPDATE shopping_carts.shopping_cart_items 
                    SET 
                        quantity = %s,
                        price = %s,
                        currency = %s,
                        product_image_url = COALESCE(%s, product_image_url)
                    WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                    RETURNING id, quantity, price
                """
                
                cursor.execute(update_query, (new_quantity, price, currency, product_image_url, user_id, cart_id, product_id))
            
            else:
                # Insert new item
                insert_query = """
                    INSERT INTO shopping_carts.shopping_cart_items (
                        user_id, shopping_cart_id, product_id,
                        price, quantity, currency, product_image_url
                    ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                    RETURNING id, quantity, price
                """
                
                cursor.execute(insert_query, (user_id, cart_id, product_id, price, quantity, currency, product_image_url))
            
    return f"Added {items} to the shopping cart."

In [14]:
add_to_shopping_cart(items, "user_1", "cart_1")

/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.20.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


"Added [{'product_id': 'B01DJI7TP8', 'quantity': 2}, {'product_id': 'B07M6SWKLT', 'quantity': 4}] to the shopping cart."

In [ ]:
items_2 = [
    {
        "product_id": "B0BR8SJM7N",
        "quantity": 4
    }
]

In [ ]:
add_to_shopping_cart(items_2, "user_1", "cart_1")

In [ ]:
items_3 = [
    {
        "product_id": "B09X1LDMH6",
        "quantity": 1
    }
]

In [ ]:
add_to_shopping_cart(items_3, "user_1", "cart_1")

### Get the Shopping Cart Items Tool

In [ ]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:

    """
    Retrieve all items in a user's shopping cart.
    
    Args:
        user_id: User identifier
        cart_id: Cart identifier
    
    Returns:
        List of dictionaries containing cart items
    """
    
    conn = psycopg2.connect(
        host="aws-1-eu-west-1.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.tljgwktvwcwhhxldhblk",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                SELECT 
                    product_id, price, quantity,
                    currency, product_image_url,
                    (price * quantity) as total_price
                FROM shopping_carts.shopping_cart_items 
                WHERE user_id = %s AND shopping_cart_id = %s
                ORDER BY added_at DESC
            """
        cursor.execute(query, (user_id, cart_id))

        return [dict(row) for row in cursor.fetchall()]

In [ ]:
get_shopping_cart("user_1", "cart_1")

In [ ]:
get_shopping_cart("user_1_asfadsf", "cart_1_asfasf")

### Deleting items from the Shopping Cart Tool

In [ ]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> str:

    """
    Remove an item completely from the shopping cart.
    
    Args:
        user_id: User identifier
        product_id: Product identifier to remove
        cart_id: Cart identifier
    
    Returns:
        Information about the removal of the item from the shopping cart.
    """
    
    conn = psycopg2.connect(
        host="aws-1-eu-west-1.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.tljgwktvwcwhhxldhblk",
        password="tools_user_password"
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:

        query = """
                DELETE FROM shopping_carts.shopping_cart_items
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
        cursor.execute(query, (user_id, cart_id, product_id))

        return f"Removed {product_id} from the shopping cart." if cursor.rowcount > 0 else f"Item {product_id} not found in the shopping cart."

In [ ]:
remove_from_cart("B09X1LDMH6", "user_1", "cart_1")

In [ ]:
remove_from_cart("B09JS5PSV1", "user_1", "cart_1")

In [ ]:
add_to_shopping_cart(items, "user_1", "cart_1")

In [ ]:
get_shopping_cart("user_1", "cart_1")

In [ ]:
add_to_shopping_cart(items, "user_1", "cart_1")

In [ ]:
get_shopping_cart("user_1", "cart_1")

In [ ]:
remove_from_cart("B0BR8SJM7N", "user_1", "cart_1")

In [ ]:
get_shopping_cart("user_1", "cart_1")

## Warehouse Manager Agent Tools

### Simulate availability in warehouses

### Import Dependencies

In [27]:
import psycopg2
from psycopg2.extras import RealDictCursor, execute_batch
import numpy as np
from qdrant_client import QdrantClient
import random

### Fictional Warehouses

In [25]:
warehouses = [
    {
        "warehouse_id": "DE-BER-01",
        "warehouse_location": "Berlin, Germany",
        "warehouse_name": "Berlin Distribution Center"
    },
    {
        "warehouse_id": "DE-MUN-01",
        "warehouse_location": "Munich, Germany",
        "warehouse_name": "Munich Logistics Hub"
    },
    {
        "warehouse_id": "DE-HAM-01",
        "warehouse_location": "Hamburg, Germany",
        "warehouse_name": "Hamburg North Warehouse"
    },
    {
        "warehouse_id": "FR-PAR-01",
        "warehouse_location": "Paris, France",
        "warehouse_name": "Paris Central Depot"
    },
    {
        "warehouse_id": "FR-LYO-01",
        "warehouse_location": "Lyon, France",
        "warehouse_name": "Lyon Regional Warehouse"
    },
    {
        "warehouse_id": "FR-MAR-01",
        "warehouse_location": "Marseille, France",
        "warehouse_name": "Marseille Mediterranean Hub"
    }
]

### Simulate Stock Availability for each of the warehouse

#### Retrieve all item IDs from the Amazon items Qdrant Collection

In [16]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_CLOUD_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=60
)

/Users/hang/Documents/Development/ai-eng-course/ai-eng-project/.venv/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.20.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [17]:
dummy_vector = np.zeros(1536).tolist()

In [18]:
payload = qdrant_client.query_points(
    collection_name="Amazon-items-collection-01-hybrid-search",
    query=dummy_vector,
    using="text-embedding-3-small",
    limit=1000,
    with_payload=["parent_asin"],
    with_vectors=False
)

In [19]:
payload.points

[ScoredPoint(id=405, version=3, score=0.0, payload={'parent_asin': 'B07XDPXH95'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=234, version=3, score=0.0, payload={'parent_asin': 'B071CJNFJ2'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=349, version=3, score=0.0, payload={'parent_asin': 'B00CXUT5ZG'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=271, version=3, score=0.0, payload={'parent_asin': 'B004S472EQ'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=147, version=3, score=0.0, payload={'parent_asin': 'B08K3F55GY'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=352, version=3, score=0.0, payload={'parent_asin': 'B07JLL7MZ5'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=376, version=3, score=0.0, payload={'parent_asin': 'B0741CXH71'}, vector=None, shard_key=None, order_value=None),
 ScoredPoint(id=416, version=3, score=0.0, payload={'parent_asin': 'B0BMZ7V5L1'}, vector=N

In [20]:
len(payload.points)

1000

In [21]:
parent_asin_list = [item.payload["parent_asin"] for item in payload.points]

In [22]:
parent_asin_list

['B07XDPXH95',
 'B071CJNFJ2',
 'B00CXUT5ZG',
 'B004S472EQ',
 'B08K3F55GY',
 'B07JLL7MZ5',
 'B0741CXH71',
 'B0BMZ7V5L1',
 'B07TSD6PB1',
 'B08HK5XCQ4',
 'B083V2W4SZ',
 'B01DJI7TP8',
 'B07Y1T6NWT',
 'B08D6TVVY5',
 'B07Y5LZBK4',
 'B07YFPGGM5',
 'B09HTKXXPT',
 'B00PNGM8UQ',
 'B0947572HS',
 'B08P4S4S4C',
 'B01FEDC0M8',
 'B08X155BYB',
 'B08PBM1TR4',
 'B09MCWWLBC',
 'B0182UE3I4',
 'B009XP1J26',
 'B085KVH16T',
 'B08H8MJY27',
 'B07HZ636L8',
 'B08Z44LXYL',
 'B089QBHN73',
 'B06XGCJLR6',
 'B078NNFHQ7',
 'B09M7Q8N8R',
 'B07NZNMLYC',
 'B07YY1N5XQ',
 'B08356M6N7',
 'B09HX83S9W',
 'B07MT4VJRN',
 'B01FAR9OQI',
 'B0001M2FYQ',
 'B08DNC3L4M',
 'B076T8L7Q2',
 'B0055PADN4',
 'B01N2NKI27',
 'B00HQIN4DY',
 'B08C2DVS1V',
 'B088JN2GGQ',
 'B003IT6YFK',
 'B0978LG2ZJ',
 'B076VJDWBP',
 'B00B1GVD5I',
 'B000HWQ75I',
 'B07BXJ6KH6',
 'B08T5SXW61',
 'B07D5JYGDP',
 'B07CNT5ST1',
 'B0C54BYRTF',
 'B09JVNM8PP',
 'B07ZYYYPQN',
 'B009VJLZ14',
 'B084LMFXF1',
 'B07MJPMQCH',
 'B00A7IZNNO',
 'B01CVHARWU',
 'B01DFKANQM',
 'B0C3VV6D

### Generate Synthetic availability for all items in Qdrant

In [23]:
def generate_inventory_data(warehouses, product_ids, availability_rate=0.75):
    
    inventory_records = []
    
    for warehouse in warehouses:
        for product_id in product_ids:
            # 75% chance the product is available in this warehouse
            if random.random() < availability_rate:
                total_quantity = random.randint(0, 100)
                
                # Only add to inventory if quantity > 0
                if total_quantity > 0:
                    inventory_records.append({
                        "warehouse_id": warehouse["warehouse_id"],
                        "warehouse_location": warehouse["warehouse_location"],
                        "warehouse_name": warehouse["warehouse_name"],
                        "product_id": product_id,
                        "total_quantity": total_quantity,
                        "reserved_quantity": 0  # Starting with no reservations
                    })
    
    return inventory_records

In [28]:
inventory_data = generate_inventory_data(warehouses, parent_asin_list, availability_rate=0.75)

In [29]:
inventory_data

[{'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B07XDPXH95',
  'total_quantity': 62,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B071CJNFJ2',
  'total_quantity': 47,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B00CXUT5ZG',
  'total_quantity': 20,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B004S472EQ',
  'total_quantity': 30,
  'reserved_quantity': 0},
 {'warehouse_id': 'DE-BER-01',
  'warehouse_location': 'Berlin, Germany',
  'warehouse_name': 'Berlin Distribution Center',
  'product_id': 'B08K3F55GY',
  'total_quantity': 97,
  

In [30]:
1000*0.75*6

4500.0

In [31]:
len(inventory_data)

4436

### Write synthetic data to Postgres

In [32]:
def insert_inventory_to_db(inventory_records):
   
    try:
        # Connect to the database
        conn = psycopg2.connect(
            host="aws-0-ap-southeast-2.pooler.supabase.com",
            port=5432,
            database="postgres",
            user="tools_user.vtluslyavpfwprmqmzft",
            password="ewTFDFew"
        )
        conn.autocommit = True

        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        
            # Prepare the INSERT query
            insert_query = """
            INSERT INTO warehouses.inventory 
            (warehouse_id, warehouse_location, warehouse_name, product_id, total_quantity, reserved_quantity)
            VALUES (%(warehouse_id)s, %(warehouse_location)s, %(warehouse_name)s, %(product_id)s, %(total_quantity)s, %(reserved_quantity)s)
            """
            
            # Use execute_batch for better performance with many inserts
            execute_batch(cursor, insert_query, inventory_records, page_size=100)
            
            # Commit the transaction
            conn.commit()
            
            print(f"Successfully inserted {len(inventory_records)} records into warehouses.inventory")
            
            # Close cursor and connection
            cursor.close()
            conn.close()
        
    except psycopg2.Error as e:
        print(f"Database error: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        print(f"Error: {e}")
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

In [33]:
insert_inventory_to_db(inventory_data)

Successfully inserted 4436 records into warehouses.inventory


## Test Warehouse Agent Tools

### Item Availability Check Tool

In [34]:
def check_warehouse_availability(items: list[dict]) -> dict:

    """Check availability of items across warehouses, including partial fulfillment options.
    
    Args:
        items: A list of items to check. Each item is a dictionary with keys: product_id, quantity.
        
    Returns:
        A dictionary containing:
        - can_fulfill_completely: bool indicating if all items can be fulfilled from at least one warehouse
        - warehouses_full_fulfillment: list of warehouses that can fulfill the entire order
        - warehouses_partial_fulfillment: list of warehouses with partial availability
        - unavailable_items: list of items that cannot be fulfilled from any warehouse
        - details: detailed breakdown per warehouse with availability for each item
    """
    
    conn = psycopg2.connect(
        host="aws-0-ap-southeast-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.vtluslyavpfwprmqmzft",
        password="ewTFDFew"
    )
    
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            result = {
                "can_fulfill_completely": False,
                "warehouses_full_fulfillment": [],
                "warehouses_partial_fulfillment": [],
                "unavailable_items": [],
                "details": []
            }
            
            # Check each warehouse for availability
            warehouse_query = """
                SELECT DISTINCT warehouse_id, warehouse_name, warehouse_location
                FROM warehouses.inventory
            """
            cursor.execute(warehouse_query)
            warehouses = cursor.fetchall()
            
            for warehouse in warehouses:
                warehouse_can_fulfill_all = True
                has_any_availability = False
                warehouse_details = {
                    "warehouse_id": warehouse['warehouse_id'],
                    "warehouse_name": warehouse['warehouse_name'],
                    "warehouse_location": warehouse['warehouse_location'],
                    "items": [],
                    "can_fulfill_all": False,
                    "has_partial": False
                }
                
                for item in items:
                    product_id = item['product_id']
                    requested_quantity = item['quantity']
                    
                    # Check availability in this warehouse
                    availability_query = """
                        SELECT product_id, total_quantity, reserved_quantity, available_quantity
                        FROM warehouses.inventory
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(availability_query, (warehouse['warehouse_id'], product_id))
                    inventory = cursor.fetchone()
                    
                    available_qty = inventory['available_quantity'] if inventory else 0
                    
                    item_detail = {
                        "product_id": product_id,
                        "requested": requested_quantity,
                        "available": available_qty,
                        "can_fulfill_completely": available_qty >= requested_quantity,
                        "can_fulfill_partially": available_qty > 0 and available_qty < requested_quantity
                    }
                    
                    warehouse_details["items"].append(item_detail)
                    
                    # Track if warehouse can fulfill this item completely
                    if available_qty < requested_quantity:
                        warehouse_can_fulfill_all = False
                    
                    # Track if warehouse has any availability for any item
                    if available_qty > 0:
                        has_any_availability = True
                
                # Categorize warehouse
                if warehouse_can_fulfill_all:
                    warehouse_details["can_fulfill_all"] = True
                    result["warehouses_full_fulfillment"].append({
                        "warehouse_id": warehouse['warehouse_id'],
                        "warehouse_name": warehouse['warehouse_name'],
                        "warehouse_location": warehouse['warehouse_location']
                    })
                elif has_any_availability:
                    warehouse_details["has_partial"] = True
                    result["warehouses_partial_fulfillment"].append({
                        "warehouse_id": warehouse['warehouse_id'],
                        "warehouse_name": warehouse['warehouse_name'],
                        "warehouse_location": warehouse['warehouse_location']
                    })
                
                result["details"].append(warehouse_details)
            
            # Check if any items cannot be fulfilled from any warehouse
            for item in items:
                product_id = item['product_id']
                requested_quantity = item['quantity']
                
                # Get total available quantity across all warehouses
                total_available_query = """
                    SELECT product_id, SUM(available_quantity) as total_available
                    FROM warehouses.inventory
                    WHERE product_id = %s
                    GROUP BY product_id
                """
                cursor.execute(total_available_query, (product_id,))
                total_available = cursor.fetchone()
                
                total_available_qty = total_available['total_available'] if total_available else 0
                
                if total_available_qty < requested_quantity:
                    result["unavailable_items"].append({
                        "product_id": product_id,
                        "requested": requested_quantity,
                        "total_available_across_warehouses": total_available_qty,
                        "shortage": requested_quantity - total_available_qty
                    })
            
            result["can_fulfill_completely"] = len(result["warehouses_full_fulfillment"]) > 0 and len(result["unavailable_items"]) == 0
            
            return result
            
    finally:
        conn.close()

In [35]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 3
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5
    }
]

In [36]:
availability = check_warehouse_availability(shopping_cart)

In [37]:
availability

{'can_fulfill_completely': False,
 'warehouses_full_fulfillment': [],
 'warehouses_partial_fulfillment': [],
 'unavailable_items': [{'product_id': 'B0C747Q3K1',
   'requested': 3,
   'total_available_across_warehouses': 0,
   'shortage': 3},
  {'product_id': 'B09P5NQH7Q',
   'requested': 2,
   'total_available_across_warehouses': 0,
   'shortage': 2},
  {'product_id': 'B0CFLR6R3X',
   'requested': 5,
   'total_available_across_warehouses': 0,
   'shortage': 5}],
 'details': [{'warehouse_id': 'DE-BER-01',
   'warehouse_name': 'Berlin Distribution Center',
   'warehouse_location': 'Berlin, Germany',
   'items': [{'product_id': 'B0C747Q3K1',
     'requested': 3,
     'available': 0,
     'can_fulfill_completely': False,
     'can_fulfill_partially': False},
    {'product_id': 'B09P5NQH7Q',
     'requested': 2,
     'available': 0,
     'can_fulfill_completely': False,
     'can_fulfill_partially': False},
    {'product_id': 'B0CFLR6R3X',
     'requested': 5,
     'available': 0,
     'can

In [ ]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 110
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5
    }
]

In [ ]:
availability = check_warehouse_availability(shopping_cart)

In [ ]:
availability

### Item Reservation Tool

In [ ]:
def reserve_warehouse_items(reservations: list[dict]) -> dict:
    
    """Reserve items from multiple warehouses in a single transaction.
    
    Args:
        reservations: A list of reservations. Each reservation is a dictionary with keys:
                     - warehouse_id: The warehouse to reserve from
                     - product_id: The product to reserve
                     - quantity: The quantity to reserve
        
    Returns:
        A dictionary containing:
        - success: bool indicating if all reservations were successful
        - reserved_items: list of successfully reserved items
        - failed_items: list of items that could not be reserved
    """
    
    conn = psycopg2.connect(
        host="aws-0-ap-southeast-2.pooler.supabase.com",
        port=5432,
        database="postgres",
        user="tools_user.vtluslyavpfwprmqmzft",
        password="ewTFDFew"
    )
    conn.autocommit = False  # Use transaction
    
    try:
        with conn.cursor(cursor_factory=RealDictCursor) as cursor:
            result = {
                "success": False,
                "reserved_items": [],
                "failed_items": []
            }
            
            for reservation in reservations:
                warehouse_id = reservation['warehouse_id']
                product_id = reservation['product_id']
                quantity = reservation['quantity']
                
                # Check and lock the inventory row
                check_query = """
                    SELECT warehouse_id, product_id, warehouse_name, warehouse_location, 
                           total_quantity, reserved_quantity, available_quantity
                    FROM warehouses.inventory
                    WHERE warehouse_id = %s AND product_id = %s
                    FOR UPDATE
                """
                cursor.execute(check_query, (warehouse_id, product_id))
                inventory = cursor.fetchone()
                
                if inventory and inventory['available_quantity'] >= quantity:
                    # Update inventory to reserve the items
                    update_query = """
                        UPDATE warehouses.inventory
                        SET reserved_quantity = reserved_quantity + %s
                        WHERE warehouse_id = %s AND product_id = %s
                    """
                    cursor.execute(update_query, (quantity, warehouse_id, product_id))
                    
                    result["reserved_items"].append({
                        "product_id": product_id,
                        "quantity": quantity,
                        "warehouse_id": warehouse_id,
                        "warehouse_name": inventory['warehouse_name'],
                        "warehouse_location": inventory['warehouse_location']
                    })
                else:
                    result["failed_items"].append({
                        "product_id": product_id,
                        "warehouse_id": warehouse_id,
                        "requested": quantity,
                        "available": inventory['available_quantity'] if inventory else 0,
                        "reason": "insufficient_stock" if inventory else "not_in_warehouse"
                    })
            
            # Only commit if all items were successfully reserved
            if len(result["failed_items"]) == 0:
                conn.commit()
                result["success"] = True
            else:
                conn.rollback()
                result["success"] = False
            
            return result
            
    except Exception as e:
        conn.rollback()
        raise e
    finally:
        conn.close()

In [ ]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 3,
        "warehouse_id": "FR-LYO-01"
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2,
        "warehouse_id": "FR-LYO-01"
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5,
        "warehouse_id": "FR-LYO-01"
    }
]

In [ ]:
reserve_warehouse_items(shopping_cart)

In [ ]:
shopping_cart = [
    {
        "product_id": "B0C747Q3K1",
        "quantity": 120,
        "warehouse_id": "FR-LYO-01"
    },
    {
        "product_id": "B09P5NQH7Q",
        "quantity": 2,
        "warehouse_id": "FR-LYO-01"
    },
    {
        "product_id": "B0CFLR6R3X",
        "quantity": 5,
        "warehouse_id": "FR-LYO-01"
    }
]

In [ ]:
reserve_warehouse_items(shopping_cart)